# Diniz Ch.7 — Lattice RLS error conventions and system identification

Example 7.1 及 Algorithm 7.1–7.4 比較。物理 a-priori MSE、posterior MSE 和 normalized MSE 各自呈現。

來源：`Diniz-Chap7-SysID.ipynb`。所有資料與結果留在記憶體，圖表使用共用深色風格。

執行順序：設定 → 參數與資料 → 模型與訓練 → 圖表與數值摘要。

## 1. Setup

In [ ]:
from pathlib import Path
import sys
from functools import partial

import numpy as np

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "adsp").is_dir())
if str(project_root) not in sys.path:
  sys.path.insert(0, str(project_root))

from adsp import data, training, statistics as stats
from adsp import plotting as plot
from adsp import filters as af
from adsp import equalization as ae
from adsp.datasets import SampleBatch

plot.use_dark()

## 2. Shared colored-input plant

In [ ]:
from adsp.experiments import diniz

n_samples, n_trials, seed, noise_var = 600, 50, 1, 1e-4
# Original ensemble size was 200; raise n_trials when reproducing its smoother curves.
make_data = partial(data.system_identification, n_samples, h=diniz.SYSTEM_TAPS, pole=0.6894, noise_var=noise_var)
models = {
  "LRLS 7.1": partial(af.LatticeRLS, filter_order=7, lam=0.99, epsilon=0.1),
  "Normalized LRLS 7.2": partial(af.NormalizedLatticeRLS, filter_order=7, lam=0.99, epsilon=0.01),
  "Error-feedback LRLS 7.3": partial(af.ErrorFeedbackLatticeRLS, filter_order=7, lam=0.99, epsilon=0.1),
  "A priori LRLS 7.4": partial(af.APrioriLatticeRLS, filter_order=7, lam=0.99, epsilon=0.1),
}

## 3. Run and keep units explicit

In [ ]:
results = training.compare(models, make_data, n_trials=n_trials, seed=seed)
plot.learning(results, noise_floor=noise_var, window=21, title="Physical a-priori MSE")
plot.show()
# Squared errors are averaged per trial; physical and normalized units stay separate.
r71 = results["LRLS 7.1"]
r72 = results["Normalized LRLS 7.2"]
fig, grid = plot.axes(rows=2)
plot.lines({
  "Posterior |ε|²": np.abs(r71.extras["e_posterior"])**2,
  "A priori |e|²": r71.mse
}, ax=grid[0, 0], scale="log", window=21, ylabel="Physical squared error", title="Ensemble LRLS physical errors")
plot.lines({"Normalized |ē|²": np.abs(r72.extras["e_normalized"])**2}, ax=grid[1, 0], scale="log", window=21,
           ylabel="Dimensionless normalized error", title="Normalized error: separate units")
plot.show(fig)
training.summary(results, tail=300, noise_floor=noise_var)
print("Theoretical input spread:", stats.eigenvalue_spread(0.6894**np.abs(np.subtract.outer(np.arange(8),
                                                                                            np.arange(8)))))